# ToolUse Agent using Anthropic models

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction TB

    INIT --> HAS_MESSAGE
    HAS_MESSAGE --> CHAT: condition_true
    HAS_MESSAGE --> TOOL_USE: condition_false

    CHAT--> IS_TOOL_CALL
    
    IS_TOOL_CALL --> FINAL: condition_false
    IS_TOOL_CALL --> TOOL_USE: condition_true

    TOOL_USE --> FINAL


```

### State Diagram (User View)
```mermaid
stateDiagram-v2
direction TB
    state "Start Agent" as StartAgent
    state "Continue Agent" as ContinueAgent
    state "Interrupt Agent" as InterruptAgent

    Initialize --> StartAgent : start_async
    StartAgent --> ContinueAgent : continue_async
    ContinueAgent --> InterruptAgent : interrupt_async
    InterruptAgent --> ContinueAgent : continue_async
    ContinueAgent --> ContinueAgent : continue_async
    ContinueAgent --> Terminate
    Terminate --> StartAgent : start_async
    Terminate --> InterruptAgent : interrupt_async

```



### a) Create Agent

In [1]:
import os
os.environ["LOG_LEVEL"] = "DEBUG"

In [2]:
import os
os.environ["LOG_LEVEL"] = "DEBUG"

from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper

from gai.lib.tests import make_local_tmp
import os
here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
from gai.messages import FileMonologue
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)

aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()

# Create an artificial dialogue history for testing
from gai.messages import FileDialogue, MessagePydantic
messages = [
    MessagePydantic(**{
        'id': 'b1e5f98c-f6eb-47de-a6e2-387510d970f9', 
        'header': {
            'sender': 'User',
            'recipient': 'Sara',
            "timestamp": 1751308157.270983,
            "order": 0            
        }, 'body': {
            'type': 'chat.send',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0,
            'step_no': 0,
            'role': "user",
            'content': 'It is a very nice weather in Singapore right now.',
        }
    }),
    MessagePydantic(**{
        'id': 'abbc7961-45dc-4973-aaf4-a6224ed35d37', 
        'header': {
            'sender': 'Sara',
            'recipient': 'User',
            "timestamp": 1751308167.3488164,
            "order": 1
        }, 'body': {
            'type': 'chat.reply',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0, 
            'step_no': 1,
            'chunk_no':10,
            'chunk':'<eom>',
            'role': "assistant",
            'content': 'Yes, it is! The weather in Singapore is typically warm and humid, with occasional rain showers. It\'s a great time to enjoy outdoor activities or relax indoors with a cool drink. How can I assist you today?'
        }
    })]
from gai.lib.constants import DEFAULT_GUID
file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(messages=messages,file_path=file_path)
recap = dialogue.extract_recap()

agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue,
    recap=recap
)

def print_chunk(chunk):
    """
    Helper function to print the chunk of data received from the agent.
    """
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
        else:
            if isinstance(chunk, list):
                for item in chunk:
                    if item.get("name"):
                        print(f'Tool: "{item["name"]}"')
                    if item.get("input"):
                        inputs = item.get("input")
                        if isinstance(inputs, dict):
                            for key, value in inputs.items():
                                if isinstance(value, str):
                                    if len(value) > 100:
                                        print(
                                            f"\tInput: {key} = {value[:100]}... (truncated)"
                                        )
                                    else:
                                        print(f"\tInput: {key} = {value}")

INFO     MessageStore: Created new message directory                                                               
         /workspaces/hydrogen-mvp/gai-sdk/test/unittest/gai/2_asm/agents/tmp

DEBUG    MessageStore: Dialogue file reset. all messages cleared.

INFO     MessageStore: Created new message store                                                                   
         /workspaces/hydrogen-mvp/gai-sdk/test/unittest/gai/2_asm/agents/tmp

INFO     mcp_client.__init__: server command=python                                                                
         args=['/workspaces/hydrogen-mvp/gai-sdk/src/gai/mcp/servers/mcp_pseudo.py']

INFO     mcp_client.__init__: server command=npx args=['time-mcp']

INFO     mcp_client.__init__: server command=uvx args=['gai-tools-web']

INFO     MessageStore: Created new message directory                                                               
         /workspaces/hydrogen-mvp/gai-sdk/test/unittest/gai/2_asm/agents/tmp

DEBUG    MessageStore: Dialogue file reset. all messages cleared.

INFO     MessageStore: Created new message store                                                                   
         /workspaces/hydrogen-mvp/gai-sdk/test/unittest/gai/2_asm/agents/tmp

### reset monologue (optional)

In [2]:
monologue.reset()


### b) start

The location of the public holiday is inferred from the dialogue context.

In [ ]:
goal = "When is the next public holiday?"
user_message=f"""
## Goal

{goal}

## Instructions

- Find the best way to achieve the goal.
"""

resp=agent.start(user_message=user_message)
# Stream the response
async for chunk in resp:
    print_chunk(chunk)

# Update dialogue
dialogue.add_user_message(recipient="Sara", content=goal)
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)

I'll help you find the next public holiday. To provide accurate information, I need to search for current public holiday information since this varies by location and changes over time.
Tool: "search"
	Input: search_query = next public holiday 2024 2025 upcoming


### c) Show monologue

In [5]:
import json

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_messages()
for message in messages[-2:]:
    print(json.dumps(message.model_dump(), indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size = agent.fsm.monologue.get_total_size()
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "id": "c3ade633-f3a9-4afe-8a98-9fb8a0fde0e3",
    "header": {
        "sender": "User",
        "recipient": "ToolUseAgent",
        "timestamp": 1752890580.7465115,
        "order": 2
    },
    "body": {
        "type": "monologue",
        "state_name": "TOOL_USE",
        "step_no": 4,
        "content_type": "text",
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": "toolu_012xou8iZjRtz1ogiEZjhBHp",
                "content": "{\n  \"query\": \"next public holiday 2024 2025 upcoming\",\n  \"chunks\": [\n    {\n      \"index\": 0,\n      \"link_title\": \"https://www.mom.gov.sg/employment-practices/public-holidays\",\n      \"link\": \"https://www.mom.gov.sg/employment-practices/public-holidays\",\n      \"chunk\": \"The next public holiday is 9 August, Saturday National Day 2024 2025 2026 Date Day Holiday image Holiday name 1 Janu

### d) resume

In [6]:
resp = agent.resume()
# Stream the response
can_resume=False
async for chunk in resp:
    can_resume=True
    print_chunk(chunk)
if not can_resume:
    print("Cannot resume because user has not provided input.")


Let me search for more current and location-specific information about upcoming public holidays:
Tool: "search"
	Input: search_query = next public holiday July 2025 August 2025 upcoming United States Canada UK


### e) User interrupt agent

Should be able to interrupt the agent and continue with original task.

In [7]:
user_message = "Tell me a one paragraph joke."
resp = agent.interrupt(user_message=user_message)
# Stream the response
async for chunk in resp:
    print_chunk(chunk)

# Update dialogue when user interrupt agent
dialogue.add_user_message(recipient="Sara", content=user_message)
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)

Here's a one-paragraph joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around in terror, only to see the librarian pointing to the shelf directly behind him. Embarrassed, he grabs a book and sits down to read, but every few minutes he keeps glancing over his shoulder. After an hour, he approaches the librarian again and says, "Excuse me, but I think someone's been watching me read." The librarian nods knowingly and replies, "Yes, that's the book's author - he's been dead for 50 years, but he always keeps an eye on his readers." The man drops the book and runs out screaming, while the librarian chuckles and mutters, "Works every time - best way to get people to actually finish reading the books they check out!"

Now, let me get back on track to help you find the next public holiday. Since I found information about Singapore's holidays but need to determine your location for the most accurate answ

MessagePydantic(id='16761a10-2e2f-4164-8689-baba087b6f0e', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1752890719.7526755, order=3), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=1, step_no=1, message_id='00000000-0000-0000-0000-000000000000.215', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content="Perfect! Based on the search results and today's date (July 19, 2025), I can now provide you with the next public holiday:\n\n## The Next Public Holiday\n\nBased on the current date (July 19, 2025), the **next public holiday** depends on your location:\n\n**Upcoming holidays in the next few weeks:**\n- **Belgium Independence Day** - July 21, 2025 (in 2 days)\n- **Peru Independence Day** - July 28, 2025 (in 9 days)  \n- **Switzerland National Day** - August 1, 2025 (in 13 days)\n- **Various countries' Emancipation Day** - August 1, 2025 (in 13 days)\n- **Singapore National Day** - A

### f) Show dialogue

In [8]:
for msg in dialogue.list_messages():
    print(f"{msg.header.sender}: {msg.body.content}")

User: Sara, When is the next public holiday?
Sara: Let me get the current date first to determine which holiday is actually the next one:
User: Sara, Tell me a one paragraph joke.
Sara: Perfect! Based on the search results and today's date (July 19, 2025), I can now provide you with the next public holiday:

## The Next Public Holiday

Based on the current date (July 19, 2025), the **next public holiday** depends on your location:

**Upcoming holidays in the next few weeks:**
- **Belgium Independence Day** - July 21, 2025 (in 2 days)
- **Peru Independence Day** - July 28, 2025 (in 9 days)  
- **Switzerland National Day** - August 1, 2025 (in 13 days)
- **Various countries' Emancipation Day** - August 1, 2025 (in 13 days)
- **Singapore National Day** - August 9, 2025 (in 21 days)
- **Japan Mountain Day** - August 11, 2025 (in 23 days)
- **Various countries' Assumption Day** - August 15, 2025 (in 27 days)

If you're in a major English-speaking country like the **United States**, the next

---
## Scenario 1 - Agent interrupts itself and User has no input

- LLM Interrupts itself to ask user question
- User cannot continue because LLM is waiting for user input

In [9]:
import os
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.lib.tests import make_local_tmp
from gai.messages import FileMonologue

here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)
aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue,
    aggregated_client=aggregated_client,
)

In [10]:
monologue.reset()

goal = "What time is it right now?"
user_message = f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
- Use "user_input" to ask for timezone before you start.
"""

resp = agent.start(user_message=user_message)
# Stream the response
async for chunk in resp:
    print_chunk(chunk)


I'll help you get the current time! First, let me ask you about your timezone preference.
Tool: "user_input"
I see you'd like to know the current time. To provide you with the most accurate time, could you please let me know which timezone you'd like the time displayed in? 

For example, you could specify:
- America/New_York (Eastern Time)
- America/Los_Angeles (Pacific Time) 
- Europe/London (GMT/BST)
- Asia/Tokyo (Japan Time)
- UTC (Coordinated Universal Time)

Or if you're not sure of the exact timezone name, just let me know your city or region and I can help determine the appropriate timezone.

What timezone would you prefer?


Confirm that "resume" returns nothing because the agent is waiting for user input.

In [11]:
resp = agent.resume()
# Stream the response
can_resume = False
async for chunk in resp:
    can_resume = True
if not can_resume:
    print("\nAgent cannot resume because User did not provide input.")
assert not can_resume, "Agent should not be able to resume while waiting for input."



Agent cannot resume because User did not provide input.


---
## Scenario 2: LLM Interrupts itself and User has input

- LLM Interrupts itself to ask user question
- User responds with answer

In [12]:
import os
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.lib.tests import make_local_tmp
from gai.messages import FileMonologue

here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)
aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue,
    aggregated_client=aggregated_client,
)

In [13]:
monologue.reset()

goal = "What time is it right now?"
user_message = f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
- Use "user_input" to ask for timezone before you start.
"""

resp = agent.start(user_message=user_message)
# Stream the response
async for chunk in resp:
    print_chunk(chunk)


I'll help you get the current time! First, let me ask you about your timezone preference.
Tool: "user_input"
What timezone would you like me to show the current time in? Please provide the timezone (for example: America/New_York, Europe/London, Asia/Tokyo, America/Los_Angeles, etc.) or just let me know your location and I can determine the appropriate timezone.
Tool: "user_input"


Confirm user cannot continue because LLM is waiting for user input.

In [14]:
resp = agent.resume()
# Stream the response
can_resume = False
async for chunk in resp:
    can_resume = True
if not can_resume:
    print("\nAgent cannot resume because User did not provide input.")
assert not can_resume, "Agent should not be able to resume while waiting for input."



Agent cannot resume because User did not provide input.


Confirm that "resume" with message will work when user responds.

In [15]:
resp = agent.resume(user_message="Use SGT")
# Stream the response
can_resume = False
async for chunk in resp:
    can_resume = True
    print_chunk(chunk)
assert can_resume, "Agent should be able to resume after user provides input."


Perfect! I'll get the current time in Singapore Time (SGT). Let me fetch that for you.
Tool: "current_time"
	Input: format = YYYY-MM-DD HH:mm:ss
	Input: timezone = Asia/Singapore


User can resume normally.

In [16]:
resp = agent.resume()
async for chunk in resp:
    print_chunk(chunk)

The current time in Singapore Time (SGT) is:

**July 19, 2025 at 10:08:27 AM**

(2025-07-19 10:08:27)
